# Generating Feature and Label Tiles from Classified LiDAR and Vector Data

This notebook demonstrates how to process a large collection of classified LiDAR files (.las / .laz) into individual feature tiles derived from LiDAR data, and how to generate corresponding raster label tiles from vector data.  
Additional preprocessing steps such as High Pass Median Filtering (HPMF) and rasterization with vector buffering are applied to enhance terrain features and create realistic training datasets for machine learning.

## Workflow:

---

1. **File search**  
   Recursively search for all `.las` and `.laz` files in the working directory or sub-folders. If the data are in zip file, they are first extracted
   
---

2. **DEM parameters**  
   Define interpolation settings such as resolution, output type (IDW), search radius, power parameter, and window size.
   
---

3. **Per-file processing**  
   For each LiDAR file:
   - Read only ground-classified points (classification code = 2).  
   - Apply statistical outlier removal to clean the point cloud.  
   - Interpolate to raster with Inverse Distance Weighting (IDW), creating one Digital Elevation Model (DEM) tile per input file.
     
---

4. **High Pass Median Filter (HPMF)**  
   Apply a high-pass median filter to the DEM tiles to highlight local elevation changes.  
   The method subtracts each cell value from the median of its neighbourhood, emphasizing fine-scale terrain variability while suppressing broad trends.
   
---

5. **Global robust statistics computation**   
    To ensure consistent and robust scaling across all tiles:
   - Each raster tile is read block by block to avoid memory overload.
   - Invalid values (e.g., NaN, -9999) are masked out.
   - Up to a fixed number (e.g., 5000) of valid pixels are randomly sampled per tile.
   - The sampled values from all tiles are aggregated into a global pool.
   - From this global pool, robust statistics are computed: Lower percentile (p1), Upper percentile (p99), Median.
   These global values are then used for consistent scaling or normalization of all tiles, ensuring that extreme outliers in individual tiles do not distort the global distribution.

---

6. **Rasterization**  
   Each DEM tile is used as a mask to extract the matching area from a large vector dataset and rasterize it into a label tile.  
   Before rasterization, vector geometries are buffered (1.5 m) to ensure coverage of narrow or thin features.  
   The buffered geometries are then rasterized onto the DEM grid.  

   To avoid creating unrealistic fixed-width labels, the rasterized geometries are combined with the HPMF output:  
   only pixels within the buffered vector lines **and** with an HPMF value below –0.075 are kept as ditch pixels.
   This ensures that labels follow real terrain depressions rather than forming uniform strips around the vector lines **(we can try different thresholds)**.

   Finally, a majority filter is applied to remove isolated spurious pixels and smooth the label shapes.  
   
---

7. **Output**  
   Results are written to dedicated subdirectories:  
   - `model_input_data/dem_tiles` -> DEM rasters generated from IDW interpolation  
   - `model_input_data/hpmf_tiles` -> DEM rasters after hierarchical progressive morphological filtering  
   - `model_input_data/rasters` -> Final standardized rasterized tiles prepared for machine learning

    
    These standardized tiles can later be mosaicked or subdivided further into training patches for the ML workflow.

---

The tile-based approach is well-suited for handling very large LiDAR datasets, since each file is processed independently without overloading RAM.


## Environment Setup and Imports

On Windows I recommend to use a dedicated **Conda environment** for this workflow, because installing **PDAL** and its dependencies can be problematic on Windows.
With Conda, installation is much simpler since most geospatial libraries (PDAL, GDAL, etc.) are available via the `conda-forge` channel.

Example environment creation:

```bash
conda create -n lidar-env python=3.11 -c conda-forge pdal numpy jupyter pathlib json
conda activate lidar-env
```

Once the environment is active, you can import the necessary Python libraries in the notebook:

In [6]:
import os
import pdal
import zipfile
import geopandas as gpd
from shapely.geometry import box
import rasterio
from rasterio import features
import numpy as np
import math
import random
import csv
from rasterio.windows import Window
from whitebox.whitebox_tools import WhiteboxTools
from tqdm import tqdm 
import json
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from pathlib import Path

wbt = WhiteboxTools()
wbt.verbose = False

### LiDAR Preprocessing and DEM Generation

File search

In [2]:
# Current working directory absolute path
curr_dir = Path().resolve()
# Data directory as a sub-folder (where I have the data, change if needed)
lidar_data_dir = curr_dir / "data" / "lidar_data"
# Recursively find all ZIP files
zip_files = list(lidar_data_dir.rglob("*.zip"))
print(f"Found {len(zip_files)} ZIP files.")
for zip_path in zip_files:
    # Create a subfolder with the same name as the ZIP file (without extension)
    extract_dir = zip_path.parent / zip_path.stem
        # Skip if already extracted
    if extract_dir.exists():
        print(f"⏩ Skipping {zip_path.name} (already extracted)")
        continue
    # Extract all contents of the ZIP file
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    print(f"Extracted: {zip_path.name} → {extract_dir}")
    
# Recursive search of *.laz or *.las files in all sub-folders
las_files = list(lidar_data_dir.rglob("*.laz")) + list(lidar_data_dir.rglob("*.las"))
las_files = [str(f) for f in las_files]
print(f"Found {len(las_files)} files")

Found 7 ZIP files.
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250904T101359216523441.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250904T102630519832085.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250904T104027841004545.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250908T075609555918793.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250908T081302777824096.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250908T082522557653894.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250908T092819382121549.zip (already extracted)
Found 1400 files


Define DEM parameters

In [3]:
# DEM parameters
resolution = 0.5
output_type = "min" # or "idw", but "min" probably better for our purpose
radius = 1.0 # 1 m, we can try different values
power = 2.0 # only for idw
window_size = 5 # if there are no points in radius, how many surrounding values use for interpolation

Set output directiories

In [4]:
data_dir = lidar_data_dir.parent
dem_dir = data_dir / "dem_data"
hpmf_dir = data_dir /  "hpmf_tiles"
label_dir = data_dir / "label_tiles"
normalized_dir = data_dir / "normalized_tiles"

# Create data folders if they don't exist
dem_dir.mkdir(parents=True, exist_ok=True)
hpmf_dir.mkdir(parents=True, exist_ok=True)
label_dir.mkdir(parents=True, exist_ok=True)
normalized_dir.mkdir(parents=True, exist_ok=True)

Process each file using PDAL pipeline and save as DEM tiles

In [24]:
# For each file (enumerate just so we can track how many files has been processed)
for i, las in enumerate(las_files, 1):
    las_path = Path(las)
    # name of DEM file
    dem_file = dem_dir / f"{las_path.stem}_dem_{output_type}.tif"
    # Tracking progress
    print(f"[{i}/{len(las_files)}] Processing {las_path.name} → {dem_file.name}")

    # input for PDAL is a JSON which can be done as a dictionary in python and then converting to JSON
    pipeline_dict = {
        "pipeline": [{"type": "readers.las", "filename": str(las_path)}, # read file     
            {"type": "filters.range", "limits": "Classification[2:2]"},  # only ground class
            {"type": "filters.outlier", "method": "statistical", "mean_k": 8, "multiplier": 2.5}, # filter outlying points
            {
                "type": "writers.gdal",  # create DEM with chosen parameters
                "filename": str(dem_file),
                "resolution": resolution,
                "output_type": output_type,
                "radius": radius,
                "power": power,
                "window_size": window_size,
                "gdaldriver": "GTiff"
            }
        ]
    }
    # convert dictionary to JSON and run pipeline
    pipeline = pdal.Pipeline(json.dumps(pipeline_dict))
    count = pipeline.execute()

print(f"Pipeline finished. Created {len(las_files)} DEM files")

[1/1400] Processing P4434E3_8.laz → P4434E3_8_dem_min.tif


KeyboardInterrupt: 

In [7]:
def run_pdal(las):
    las_path = Path(las)
    dem_file = dem_dir / f"{las_path.stem}_dem_{output_type}.tif"
    if dem_file.exists():
        return f"⏩ {las_path.name} skipped"

    # Create pipeline JSON dynamically
    pipeline_dict = {
        "pipeline": [
            {"type": "readers.las", "filename": str(las_path)},
            {"type": "filters.range", "limits": "Classification[2:2]"},
            {"type": "filters.outlier", "method": "statistical", "mean_k": 8, "multiplier": 2.5},
            {
                "type": "writers.gdal",
                "filename": str(dem_file),
                "resolution": resolution,
                "output_type": output_type,
                "radius": radius,
                "power": power,
                "window_size": window_size,
                "gdaldriver": "GTiff"
            }
        ]
    }

    # Save temporary pipeline file
    temp_json = dem_dir / f"{las_path.stem}_pipeline.json"
    with open(temp_json, "w") as f:
        json.dump(pipeline_dict, f)

    # Run PDAL using the JSON pipeline
    cmd = ["pdal", "pipeline", str(temp_json)]
    result = subprocess.run(cmd, capture_output=True, text=True)

    # Clean up JSON file
    temp_json.unlink(missing_ok=True)

    if result.returncode != 0:
        return f"{las_path.name} failed:\n{result.stderr.strip()}"
    else:
        return f"{las_path.name} done"


# parallel execution
max_workers = max(1, os.cpu_count() - 1)
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(run_pdal, las) for las in las_files]

    for f in tqdm(as_completed(futures), total=len(futures), desc="Processing DEMs"):
        tqdm.write(f.result())


⏩ P4434F1_3.laz skipped
⏩ P4434F2_3.laz skipped
⏩ P4434F2_2.laz skipped
⏩ P4434F1_1.laz skipped
⏩ P4434F4_2.laz skipped
⏩ P4434G2_8.laz skipped
⏩ P4434F2_5.laz skipped
⏩ P4434F4_1.laz skipped
⏩ P4434F1_8.laz skipped
⏩ P4434F1_7.laz skipped
⏩ P4434E4_1.laz skipped
⏩ P4434G1_8.laz skipped
⏩ P4434F1_4.laz skipped
⏩ P4434E4_3.laz skipped
⏩ P4434E3_8.laz skipped
⏩ P4434E4_2.laz skipped
⏩ P4434F3_9.laz skipped
⏩ P4434E4_5.laz skipped
⏩ P4434F2_6.laz skipped
⏩ P4434E4_8.laz skipped
⏩ P4434F3_5.laz skipped
⏩ P4434F3_2.laz skipped
⏩ P4434F2_1.laz skipped
⏩ P4434F3_3.laz skipped
⏩ P4434F1_5.laz skipped
⏩ P4434G1_2.laz skipped
⏩ P4434F4_9.laz skipped
⏩ P4434G1_9.laz skipped


Processing DEMs:   5%|▌         | 74/1400 [00:00<00:04, 273.85it/s]

⏩ P4434F3_1.laz skipped
⏩ P4434F4_4.laz skipped
⏩ P4434F4_5.laz skipped
⏩ P4434G1_1.laz skipped
⏩ P4434G1_7.laz skipped
⏩ P4434F4_8.laz skipped
⏩ P4434F2_4.laz skipped
⏩ P4434E4_4.laz skipped
⏩ P4434F2_7.laz skipped
⏩ P4434F3_4.laz skipped
⏩ P4434G2_1.laz skipped
⏩ P4434F3_6.laz skipped
⏩ P4434F1_6.laz skipped
⏩ P4434G1_3.laz skipped
⏩ P4434F1_2.laz skipped
⏩ P4434G1_4.laz skipped
⏩ P4434E4_6.laz skipped
⏩ P4434F2_8.laz skipped
⏩ P4434G2_4.laz skipped
⏩ P4434F2_9.laz skipped
⏩ P4434F1_9.laz skipped
⏩ P4434F4_3.laz skipped
⏩ P4434F4_7.laz skipped
⏩ P4434E3_9.laz skipped
⏩ P4434F3_8.laz skipped
⏩ P4434G2_5.laz skipped
⏩ P4434E4_7.laz skipped
⏩ P4434F3_7.laz skipped
⏩ P4434F4_6.laz skipped
⏩ P4434G2_3.laz skipped
⏩ P4434E4_9.laz skipped
⏩ P4434G1_6.laz skipped
⏩ P4434G2_2.laz skipped
⏩ P4434G2_6.laz skipped
⏩ P4434G2_7.laz skipped
⏩ P4434G1_5.laz skipped
⏩ P4434G2_9.laz skipped
⏩ P4434G3_1.laz skipped
⏩ P4434G3_5.laz skipped
⏩ P4434G3_6.laz skipped
⏩ P4434G3_7.laz skipped
⏩ P4434G3_8.laz 

Processing DEMs:  10%|█         | 143/1400 [00:00<00:04, 313.40it/s]

⏩ P4434H2_5.laz skipped
⏩ P4434H2_6.laz skipped
⏩ P4434H3_3.laz skipped
⏩ P4434H2_7.laz skipped
⏩ P4434H4_1.laz skipped
⏩ P4434H2_8.laz skipped
⏩ P4434H3_6.laz skipped
⏩ P4434H3_7.laz skipped
⏩ P4434H3_8.laz skipped
⏩ P4434H3_9.laz skipped
⏩ P4434H4_6.laz skipped
⏩ P4434H3_5.laz skipped
⏩ P4434H4_2.laz skipped
⏩ P4434H4_4.laz skipped
⏩ P4434H4_5.laz skipped
⏩ P4441G1_2.laz skipped
⏩ P4434H4_7.laz skipped
⏩ P4434H4_8.laz skipped
⏩ P4434H4_9.laz skipped
⏩ P4434H4_3.laz skipped
⏩ P4441G1_3.laz skipped
⏩ P4441G1_4.laz skipped
⏩ P4441G1_5.laz skipped
⏩ P4441G1_6.laz skipped
⏩ P4441G1_7.laz skipped
⏩ P4441G1_1.laz skipped
⏩ P4441G1_8.laz skipped
⏩ P4441G1_9.laz skipped
⏩ P4441G2_1.laz skipped
⏩ P4441G2_4.laz skipped
⏩ P4441G3_3.laz skipped
⏩ P4441G2_8.laz skipped
⏩ P4441G2_9.laz skipped
⏩ P4441G3_2.laz skipped
⏩ P4441G2_7.laz skipped
⏩ P4441G3_4.laz skipped
⏩ P4441G3_5.laz skipped
⏩ P4441G3_6.laz skipped
⏩ P4441G4_1.laz skipped
⏩ P4441G4_3.laz skipped
⏩ P4441G3_7.laz skipped
⏩ P4441G3_1.laz 

...

⏩ P4431H3_4.laz skipped
⏩ P4431H3_5.laz skipped
⏩ P4431H3_8.laz skipped
⏩ P4431H3_9.laz skipped
⏩ P4431H4_1.laz skipped
⏩ P4431H4_2.laz skipped
⏩ P4431H4_3.laz skipped
⏩ P4431H4_7.laz skipped
⏩ P4431H4_4.laz skipped
⏩ P4431H4_5.laz skipped
⏩ P4432G1_1.laz skipped
⏩ P4431H3_6.laz skipped
⏩ P4432G1_3.laz skipped
⏩ P4431H4_9.laz skipped
⏩ P4431H3_7.laz skipped
⏩ P4432G1_2.laz skipped
⏩ P4431H4_8.laz skipped
⏩ P4432G1_4.laz skipped
⏩ P4432G1_9.laz skipped
⏩ P4431H4_6.laz skipped
⏩ P4432G1_7.laz skipped
⏩ P4432G1_8.laz skipped
⏩ P4432G1_5.laz skipped
⏩ P4432G2_1.laz skipped
⏩ P4432G1_6.laz skipped
⏩ P4432G2_3.laz skipped
⏩ P4432G2_4.laz skipped
⏩ P4432G2_5.laz skipped
⏩ P4432G2_6.laz skipped
⏩ P4432G2_2.laz skipped
⏩ P4432G3_4.laz skipped
⏩ P4432G2_7.laz skipped
⏩ P4432G2_8.laz skipped
⏩ P4432G2_9.laz skipped
⏩ P4432G3_1.laz skipped
⏩ P4432G3_8.laz skipped
⏩ P4432G3_3.laz skipped
⏩ P4432G3_6.laz skipped
⏩ P4432G3_7.laz skipped
⏩ P4432G3_2.laz skipped
⏩ P4432G3_9.laz skipped
⏩ P4432G4_1.laz 

Processing DEMs:  98%|█████████▊| 1372/1400 [00:03<00:00, 385.66it/s]

⏩ P4432H2_2.laz skipped
⏩ P4432H2_5.laz skipped
⏩ P4432H2_7.laz skipped
⏩ P4432H2_8.laz skipped
⏩ P4432H1_6.laz skipped
⏩ P4432H2_3.laz skipped
⏩ P4432H2_4.laz skipped
⏩ P4432H3_4.laz skipped
⏩ P4432H3_5.laz skipped
⏩ P4432H3_6.laz skipped
⏩ P4432H3_1.laz skipped
⏩ P4432H3_2.laz skipped
⏩ P4432H3_3.laz skipped
⏩ P4432H2_6.laz skipped
⏩ P4432H2_9.laz skipped
⏩ P4432H3_7.laz skipped
⏩ P4432H4_4.laz skipped
⏩ P4432H3_9.laz skipped
⏩ P4432H4_1.laz skipped
⏩ P4432H4_2.laz skipped
⏩ P4432H3_8.laz skipped
⏩ P4432H4_5.laz skipped
⏩ P4432H4_6.laz skipped
⏩ P4433A1_2.laz skipped
⏩ P4432H4_8.laz skipped
⏩ P4432H4_3.laz skipped
⏩ P4432H4_9.laz skipped
⏩ P4433A1_1.laz skipped
⏩ P4432H4_7.laz skipped
⏩ P4433A1_3.laz skipped
⏩ P4433A1_4.laz skipped
⏩ P4433A1_5.laz skipped
⏩ P4433A1_6.laz skipped
⏩ P4433A1_7.laz skipped
⏩ P4433A1_9.laz skipped
⏩ P4433A2_1.laz skipped
⏩ P4433A2_2.laz skipped
⏩ P4433A2_3.laz skipped
⏩ P4433A2_4.laz skipped
⏩ P4433A2_9.laz skipped
⏩ P4433A1_8.laz skipped
⏩ P4433A2_8.laz 

Processing DEMs: 100%|██████████| 1400/1400 [00:03<00:00, 350.95it/s]

⏩ P4433B2_3.laz skipped
⏩ P4433B1_6.laz skipped
⏩ P4433A4_7.laz skipped
⏩ P4433B1_9.laz skipped
⏩ P4433B2_1.laz skipped
⏩ P4433B2_2.laz skipped
⏩ P4433B1_5.laz skipped
⏩ P4433B2_4.laz skipped
⏩ P4433B2_6.laz skipped
⏩ P4433B2_7.laz skipped
⏩ P4433B3_1.laz skipped
⏩ P4433B3_2.laz skipped
⏩ P4433B3_4.laz skipped
⏩ P4433B3_7.laz skipped
⏩ P4433B3_9.laz skipped
⏩ P4433B4_9.laz skipped


### Feature Enhancement and Label Generation (HPMF & Rasterization)

In [6]:
# Load vector data (ditch lines) from GeoPackage
# !!! IMPORTANT: Replace with the path to your own vector dataset !!!
label_vector_gdf = gpd.read_file("./data/vector_data/Hytky_iisalmi.gpkg")

In [7]:
# Iterate through all DEM tiles
for dem in dem_dir.iterdir():
    # Apply High Pass Median Filter (HPMF) to DEM
    hpmf_file = hpmf_dir / f"{dem.stem}_hpmf.tif"
    wbt.high_pass_median_filter(i=dem, output=hpmf_file, filterx=11, filtery=11)

    # Open the HPMF raster and read array + metadata
    with rasterio.open(hpmf_file) as hpmf_raster:
        hpmf_array = hpmf_raster.read(1)       # Read raster values as array
        hpmf_bounds = hpmf_raster.bounds       # Get raster bounding box
        hpmf_shape = hpmf_raster.shape         # Get raster dimensions (rows, cols)
        transform = hpmf_raster.transform      # Get affine transform (pixel -> coords)

    # Create a polygon covering the HPMF tile extent
    hpmf_geom = box(hpmf_bounds.left, hpmf_bounds.bottom, hpmf_bounds.right, hpmf_bounds.top)
    hpmf_gdf = gpd.GeoDataFrame(geometry=[hpmf_geom], crs=label_vector_gdf.crs)

    # Clip vector data (ditches) to HPMF tile extent
    clipped_label_vector_gdf = gpd.clip(gdf=label_vector_gdf, mask=hpmf_gdf)

    # Buffer vector geometries (1.5 m) to give them width
    buffered_label_geom = clipped_label_vector_gdf.buffer(distance=1.5)

    # Rasterize buffered geometries onto HPMF tile grid
    buffered_label_array = features.rasterize(shapes=[(geom, 1) for geom in buffered_label_geom.geometry], # Geometries to rasterize (value=1 inside buffer)
                                              out_shape=hpmf_shape,                                        # Match output size to HPMF raster
                                              transform=transform,                                         # Align to same grid/coordinates as HPMF
                                              fill=0,                                                      # Background pixels get value 0
                                              dtype=np.uint8,                                              # Use 8-bit integer values
                                              all_touched=True)                                            # Mark all pixels touched by geometry, not just centers)

    # Combine buffered vector raster with HPMF mask
    # Keep only pixels within buffer where HPMF < -0.075 (threshold 0.00 might work better for our data)
    final_label_array = np.where((buffered_label_array == 1) & (hpmf_array < -0.075), 1, 0)

    # Save label raster aligned to DEM/HPMF tile
    label_file = label_dir / f"{dem.stem}_label.tif"

    # Update metadata for label raster
    meta = hpmf_raster.meta.copy()
    meta.update({
        "dtype": "uint8",
        "count": 1,
        "nodata": None
    })
    
    with rasterio.open(label_file, "w", **meta) as label_raster:
        label_raster.write(final_label_array.astype(rasterio.uint8), 1)

    # Apply majority filter to clean noise and smooth labels
    wbt.majority_filter(i=label_file, output=label_file, filterx=3, filtery=3)


### Global Robust Normalization

To efficiently compute global robust statistics, a subset of valid pixels is sampled from each raster tile instead of loading all data into memory. Each tile is read block by block, and invalid values (e.g., NaN, -9999) are masked out. Up to a fixed number (e.g., 5000) of valid pixels are randomly selected from each file. All sampled pixels are aggregated into a global pool, from which percentiles (p1, p99) and the median are computed. 


In [8]:
# ----------------------------------------------------------------------
# Parameters
# ----------------------------------------------------------------------
MAX_SAMPLES_PER_TILE = 5000     
P_LO, P_HI = 1, 99              
EPS = 1e-6                      
OUTPUT_NODATA = -9999.0         
NUM_WORKERS = max(1, os.cpu_count() - 1)
rng = np.random.default_rng(2025)

# ----------------------------------------------------------------------
# Utility functions
# ----------------------------------------------------------------------
def read_blocks_randomized(src):
    blocks = list(src.block_windows(1))
    rng.shuffle(blocks)
    for ji, window in blocks:             
        arr = src.read(1, window=window)   
        yield window, arr

def mask_nodata(arr, nodata_val):
    mask = np.ones(arr.shape, dtype=bool)
    if nodata_val is not None and not (isinstance(nodata_val, float) and math.isnan(nodata_val)):
        mask &= (arr != nodata_val)
    mask &= (arr > -1e3)  
    mask &= ~np.isnan(arr)
    return mask

def sample_tile_values(tif_path, per_tile_cap=MAX_SAMPLES_PER_TILE):
    samples, remaining = [], per_tile_cap
    with rasterio.open(tif_path) as src:
        nodata_val = src.nodata
        for _, block in read_blocks_randomized(src):
            block = block.astype("float32", copy=False)
            valid = mask_nodata(block, nodata_val)
            if not valid.any(): 
                continue
            vals = block[valid]
            if vals.size <= remaining:
                samples.append(vals); remaining -= vals.size
            else:
                idx = rng.choice(vals.size, size=remaining, replace=False)
                samples.append(vals[idx]); remaining = 0
            if remaining <= 0:
                break
    return np.concatenate(samples) if samples else np.empty((0,), dtype="float32")

def compute_global_params(tif_list):
    all_samples = []
    for i, p in enumerate(tif_list, 1):
        s = sample_tile_values(p)
        if s.size:
            all_samples.append(s)
        if i % 10 == 0 or i == len(tif_list):
            print(f"[Pass1] Sampled {i}/{len(tif_list)} files")
    if not all_samples:
        raise RuntimeError("No valid samples collected. Check NoData handling or input path.")
    pooled = np.concatenate(all_samples)
    g_p1 = float(np.percentile(pooled, P_LO))
    g_p99 = float(np.percentile(pooled, P_HI))
    g_med = float(np.median(pooled))
    print(f"\n[Global] p{P_LO}={g_p1:.6f}, median={g_med:.6f}, p{P_HI}={g_p99:.6f} (sample size={pooled.size:,})")
    return g_p1, g_p99, g_med

def normalize_block(block, p1, p99, med):
    out = block.astype("float32", copy=True)
    np.clip(out, p1, p99, out=out)
    scale = max(p99 - p1, EPS)
    out = (out - med) / scale
    return out

def normalize_one_file(src_path, dst_path, p1, p99, med):
    with rasterio.open(src_path) as src:
        profile = src.profile
        nodata_val = src.nodata
        profile.update(
            dtype="float32",
            count=1,
            nodata=OUTPUT_NODATA,
            compress="deflate", 
            predictor=3,         
            tiled=False,
            bigtiff="IF_SAFER"
        )
        with rasterio.open(dst_path, "w", **profile) as dst:
          for ji, window in src.block_windows(1):                 
                arr = src.read(1, window=window).astype("float32", copy=False)
                valid = mask_nodata(arr, nodata_val)
                tile = np.full(arr.shape, OUTPUT_NODATA, dtype="float32")
                if valid.any():
                    norm = normalize_block(arr, p1, p99, med)
                    tile[valid] = norm[valid]
                dst.write(tile, 1, window=window)
 


In [9]:
# ----------------------------------------------------------------------
# Main routine
# ----------------------------------------------------------------------
def run_global_robust_normalization(hpmf_dir: Path, out_dir: Path):
    tif_list = sorted(hpmf_dir.glob("*.tif"))
    if not tif_list:
        raise FileNotFoundError(f"No GeoTIFF found in: {hpmf_dir}")

    print(f"Found {len(tif_list)} files. Starting Pass 1 (sampling)...")
    p1, p99, med = compute_global_params(tif_list)

    params_path = out_dir / "global_norm_params.json"
    with params_path.open("w") as f:
        json.dump({"p1": p1, "median": med, "p99": p99, "P_LO": P_LO, "P_HI": P_HI}, f, indent=2)
    print(f"Saved global parameters to: {params_path}")

    print(f"\nStarting Pass 2 (normalize & write) -> {out_dir}")
    tasks = []
    for p in tif_list:
        dst = out_dir / f"{p.stem}_normalized.tif"
        tasks.append((p, dst))

    done = 0
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futures = {ex.submit(normalize_one_file, str(src), str(dst), p1, p99, med): (src, dst) for src, dst in tasks}
        for fut in as_completed(futures):
            _ = fut.result() 
            done += 1
            if done % 10 == 0 or done == len(tasks):
                print(f"[Pass2] Normalized {done}/{len(tasks)}")

    index_csv = normalized_dir / "normalized_index.csv"
    with index_csv.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["source", "normalized"])
        for _, dst in tasks:
            w.writerow([str(_), str(dst)])
    print(f"\nAll normalization done.")
    print(f"Index CSV: {index_csv}")
    print(f"Output dir: {normalized_dir}")

# Run
run_global_robust_normalization(hpmf_dir, normalized_dir)

Found 3 files. Starting Pass 1 (sampling)...
[Pass1] Sampled 3/3 files

[Global] p1=-0.230000, median=0.000000, p99=0.190000 (sample size=15,000)
Saved global parameters to: C:\Users\Matěj\Documents\GitHub\GIS_E6010_Project_Course_2025\preprocessing\model_input_data\normalized_tiles\global_norm_params.json

Starting Pass 2 (normalize & write) -> C:\Users\Matěj\Documents\GitHub\GIS_E6010_Project_Course_2025\preprocessing\model_input_data\normalized_tiles
[Pass2] Normalized 3/3

All normalization done.
Index CSV: C:\Users\Matěj\Documents\GitHub\GIS_E6010_Project_Course_2025\preprocessing\model_input_data\normalized_tiles\normalized_index.csv
Output dir: C:\Users\Matěj\Documents\GitHub\GIS_E6010_Project_Course_2025\preprocessing\model_input_data\normalized_tiles
